# 01 · Data exploration: raw monthly sales exports

**Purpose.** Before writing any cleaning code, inspect the raw exports and list every data-quality issue, with the number of rows affected. The cleaning pipeline (`src/sales_report/cleaning.py`) is then designed from these findings, not from guesses.

**Data.** One CSV export per month (`data/raw/monthly_sales/`, January 2025 to August 2026) plus three reference tables (`data/reference/`). The dataset is synthetic, created by `scripts/generate_data.py` with a fixed seed, so every number below is reproducible.

**How to run.** From the project root, with the virtual environment active: `python -m jupyter lab`, open this notebook, then *Kernel > Restart Kernel and Run All Cells*.

**Contents**
1. Setup
2. First look at one file
3. Load all monthly files
4. Data-quality checks
5. Summary of findings and planned fixes

## 1. Setup

In [17]:
from pathlib import Path

import pandas as pd

# Paths are relative to this notebook (notebooks/).
RAW = Path("../data/raw/monthly_sales")
REF = Path("../data/reference")

# The six official regions, used to detect spelling variants.
OFFICIAL_REGIONS = ["Benelux", "France", "Germany", "Italy", "Spain", "UK"]

pd.set_option("display.max_columns", 20)

## 2. First look at one file

Let's load and open one month with `pd.read_csv` and see what happens.

In [18]:
try:
    pd.read_csv(RAW / "sales_2025_01.csv")
except Exception as e:
    print(type(e).__name__, ":", e)

ParserError : Error tokenizing data. C error: Expected 1 fields in line 7, saw 2



**Note.** The default reading fails. `read_csv` expects commas `,` separator between columns, but the header line contains none, so pandas assumes the file has a single column. Then on line 7 of the file (order `O50296`) contains the amount `95,16`: the decimal comma is taken as a separator, giving 2 fields instead of 1.

So we've already found two problems: the **columns are separated by `;`**, and **some amounts use a comma instead of a decimal point**.

Let's read it again properly. I'll load every column as text (`dtype=str`) to see the values exactly as they were exported, and use `keep_default_na=False` so empty cells stay as `""` and we can count them later.

In [19]:
jan = pd.read_csv(RAW / "sales_2025_01.csv", sep=";", dtype=str, keep_default_na=False)
print(jan.shape)
jan.head(10)

(323, 8)


,Order ID,Order Date,Customer ID,Region,Product ID,Qty,Discount,Amount EUR
0,O50075,02/01/2025,C071,UK,P127,63,0.00,151.20
1,O50123,19/01/2025,C008,France,P118,76,0.10,480.85
2,O50126,08/01/2025,C074,France,P109,108,0.05,1011.64
3,O50028,26/01/2025,C061,France,P117,120,0.00,631.20
4,O50306,07/01/2025,C018,Benelux,P116,109,0.05,706.21
5,O50296,2025-01-11,C058,Germany,P115,13,0.00,"95,16"
6,O50313,19/01/2025,C008,France,P107,49,0.10,592.26
7,O50274,2025-01-20,C079,France,P125,55,0.00,249.70
8,O50262,2025-01-03,C030,Germany,P127,95,0.00,228.00
9,O50195,15/01/2025,C039,UK,P113,14,0.00,60.62


**Takeaway.** January 2025 has 323 rows and 8 columns. In the first 10 rows alone, dates appear in two formats (`02/01/2025` and `2025-01-11`) and order `O50296` has the amount `95,16`.

## 3. Load all monthly files

Let's read every `sales_*.csv` file and stack them into one table (the pandas equivalent of *Append* in Power Query). A `source_file` column keeps track of which file each row came from, which is handy when something looks off and helps trace any problem back to its file.

In [20]:
files = sorted(RAW.glob("sales_*.csv"))
print(f"{len(files)} files: {files[0].name} ... {files[-1].name}")

df = pd.concat(
    [
        pd.read_csv(f, sep=";", dtype=str, keep_default_na=False).assign(source_file=f.name)
        for f in files
    ],
    ignore_index=True,
)
print(df.shape)
df.info()

20 files: sales_2025_01.csv ... sales_2026_08.csv
(6078, 9)
<class 'pandas.DataFrame'>
RangeIndex: 6078 entries, 0 to 6077
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Order ID     6078 non-null   str  
 1   Order Date   6078 non-null   str  
 2   Customer ID  6078 non-null   str  
 3   Region       6078 non-null   str  
 4   Product ID   6078 non-null   str  
 5   Qty          6078 non-null   str  
 6   Discount     6078 non-null   str  
 7   Amount EUR   6078 non-null   str  
 8   source_file  6078 non-null   str  
dtypes: str(9)
memory usage: 427.5 KB


In [21]:
# Rows per file: checks that no file is empty or truncated, and shows the seasonality.
df["source_file"].value_counts().sort_index()

source_file
sales_2025_01.csv    323
sales_2025_02.csv    301
sales_2025_03.csv    283
sales_2025_04.csv    314
sales_2025_05.csv    300
sales_2025_06.csv    287
sales_2025_07.csv    272
sales_2025_08.csv    263
sales_2025_09.csv    278
sales_2025_10.csv    273
sales_2025_11.csv    385
sales_2025_12.csv    350
sales_2026_01.csv    308
sales_2026_02.csv    306
sales_2026_03.csv    340
sales_2026_04.csv    324
sales_2026_05.csv    324
sales_2026_06.csv    309
sales_2026_07.csv    252
sales_2026_08.csv    286
Name: count, dtype: int64

All 20 files loaded, 6,078 rows. rows in total. Every column is still text, as intended. Monthly volumes range from 263 to 385 rows, with November and December the busiest, and none of the files look empty or truncated. 

## 4. Data-quality checks

Each check below answers one question about the data. For the duplicate check, I compare only the columns from the original export and leave out `source_file`, so two identical order lines still count as duplicates even if they come from different files.

In [22]:
COLS = [c for c in df.columns if c != "source_file"]

### 4.1 Duplicate rows

*Why it matters:* if an order line appears twice, it's counted twice in every total, so revenue looks higher than it really is, inflating turnover.

In [23]:
dup_mask = df.duplicated(subset=COLS)
print("Exact duplicate rows:", dup_mask.sum())
print("Duplicated order IDs:", df["Order ID"].duplicated().sum())

# Show a few duplicated orders side by side
df[df["Order ID"].duplicated(keep=False)].sort_values("Order ID").head(6)

Exact duplicate rows: 80
Duplicated order IDs: 80


,Order ID,Order Date,Customer ID,Region,Product ID,Qty,Discount,Amount EUR,source_file
131,O50067,16/01/2025,C077,Benelux,P115,42,0.00,307.44,sales_2025_01.csv
281,O50067,16/01/2025,C077,Benelux,P115,42,0.00,307.44,sales_2025_01.csv
49,O50281,03/01/2025,C019,Germany,P116,26,0.05,168.45,sales_2025_01.csv
208,O50281,03/01/2025,C019,Germany,P116,26,0.05,168.45,sales_2025_01.csv
91,O50290,18/01/2025,C054,Benelux,P123,9,0.00,36.72,sales_2025_01.csv
277,O50290,18/01/2025,C054,Benelux,P123,9,0.00,36.72,sales_2025_01.csv


**Result.** There are 80 exact duplicate rows, 4 in each file. There are also exactly 80 repeated order IDs. Since each order ID should appear only once, this means every repeated ID is a full copy of another line, not a different order that happens to share the same ID.

**Fix:** remove the exact duplicates.

### 4.2 Date formats

*Why it matters:* mixed date formats are easy to get wrong. Take `03/02/2026`: is that 3 February or 2 March? Some dates may not convert at all.


In [24]:
dates = df["Order Date"]
is_ddmmyyyy = dates.str.match(r"^\d{2}/\d{2}/\d{4}$")
is_iso = dates.str.match(r"^\d{4}-\d{2}-\d{2}$")

pd.Series({
    "dd/mm/yyyy": is_ddmmyyyy.sum(),
    "yyyy-mm-dd": is_iso.sum(),
    "other format": (~is_ddmmyyyy & ~is_iso).sum(),
})

dd/mm/yyyy      5151
yyyy-mm-dd       927
other format       0
dtype: int64

In [25]:
dates[is_iso].head()

5     2025-01-11
7     2025-01-20
8     2025-01-03
12    2025-01-27
16    2025-01-21
Name: Order Date, dtype: str

**Result.** 927 dates (about 15%) use the ISO format `yyyy-mm-dd`, and all the others use `dd/mm/yyyy`, and no other format exists. 

**Fix:** convert each format explicitly instead of letting pandas guess. A guess could swap the day and the month without any error.

### 4.3 Region spellings

*Why it matters:* a pivot table by region would treat `France`, `FRANCE` and `' France '` as three different regions, so France's sales would be split across three rows. 




In [26]:
print("Distinct values:", df["Region"].nunique(), "(expected 6)")

# repr() shows quotes around each value, which makes hidden spaces visible
df["Region"].map(repr).value_counts()

Distinct values: 17 (expected 6)


Region
'UK'           1299
'Benelux'      1266
'France'        960
'Germany'       865
'Spain'         787
'Italy'         400
'BENELUX'        78
' UK '           61
' Benelux '      61
'FRANCE'         56
' France '       46
' Germany '      42
'GERMANY'        40
' Spain '        37
'SPAIN'          36
'ITALY'          26
' Italy '        18
Name: count, dtype: int64

In [11]:
canonical = {r.casefold(): r for r in OFFICIAL_REGIONS}
region_fixed = df["Region"].str.strip().str.casefold().map(canonical)

print("Rows whose region differs from the official spelling:", (region_fixed != df["Region"]).sum())
print("Values that match no official region:", region_fixed.isna().sum())

Rows whose region differs from the official spelling: 501
Values that match no official region: 0


**Result.** There are 17 different spellings instead of the 6 real regions, some in capitals (`FRANCE`) and some with extra spaces (`' France '`). This affects 501 rows. Once the spaces are removed and case is ignored, every value matches one of the official regions, so nothing is unknown.

**Fix:** remove the extra spaces, then map each value to its official name, ignoring case. A simple `.str.title()` won't work here, because it would turn `UK` into `Uk`.


### 4.4 Amounts with a decimal comma

*Why it matters:* pandas can't read `95,16` as a number. so the conversion either fails or or quietly leaves the amount empty, and that sale drops out of the totals.


In [27]:
has_comma = df["Amount EUR"].str.contains(",", regex=False)
print("Amounts with a decimal comma:", has_comma.sum())
df.loc[has_comma, ["Order ID", "source_file", "Amount EUR"]].head()

Amounts with a decimal comma: 623


,Order ID,source_file,Amount EUR
5,O50296,sales_2025_01.csv,"95,16"
23,O50251,sales_2025_01.csv,"379,62"
26,O50132,sales_2025_01.csv,"1294,84"
37,O50296,sales_2025_01.csv,"95,16"
45,O50060,sales_2025_01.csv,"1910,07"


In [28]:
# After replacing the comma, does every amount convert to a positive number?
amount = pd.to_numeric(df["Amount EUR"].str.replace(",", ".", regex=False), errors="coerce")
print("Not convertible:", amount.isna().sum())
amount.describe()

Not convertible: 0


count    6078.000000
mean      512.822578
std       494.654870
min        11.400000
25%       178.750000
50%       352.930000
75%       646.850000
max      2785.200000
Name: Amount EUR, dtype: float64

**Result.** 623 amounts (about 10%) use a comma instead of a decimal point. Once the comma is replaced with a dot, every amount converts to a number, and all of them are positive and look realistic.

**Fix:** replace `,` with `.` before converting the amounts to numbers.

### 4.5 Empty quantities

*Why it matters:* rows without a quantity break any total of units sold. The quick fix would be to delete them, but they still have an amount, so deleting them would make revenue look lower than it really is.


In [29]:
empty_qty = df["Qty"] == ""
print("Empty quantities:", empty_qty.sum())
df.loc[empty_qty, ["Order ID", "Product ID", "Discount", "Amount EUR"]].head()

Empty quantities: 119


,Order ID,Product ID,Discount,Amount EUR
107,O50066,P107,0.00,"1195,27"
166,O50168,P101,0.00,1417.65
185,O50263,P122,0.05,447.27
250,O50026,P122,0.00,529.00
257,O50078,P116,0.10,521.73


**Result.** 119 rows have no quantity, but their amount, product and discount are all there. Since *amount = quantity x unit price x (1 - discount)*, we can recover the quantity.

**Fix:** recalculate the quantity from the amount, the product's unit price and the discount, round it to a whole number, and flag these rows so we know the quantity was filled in, not exported.


### 4.6 IDs missing from the reference tables

*Why it matters:* if an order's product or customer isn't in the reference tables, we can't tell its category or segment. That order would then be missing from those breakdowns.


In [30]:
products = pd.read_csv(REF / "products.csv")    # reference files use "," as separator
customers = pd.read_csv(REF / "customers.csv")

print("Unknown product IDs:", (~df["Product ID"].isin(products["ProductID"])).sum())
print("Unknown customer IDs:", (~df["Customer ID"].isin(customers["CustomerID"])).sum())

Unknown product IDs: 0
Unknown customer IDs: 0


**Result.** Every product and customer ID in the exports exists in the reference tables.

**Fix:** nothing to fix for now. The pipeline will still check this on every run, because a future export could include a new product or customer that isn't in the reference tables yet.


## 5. Summary of findings and planned fixes

The table below is computed from the checks above, so it updates on its own if the data changes.

In [31]:
summary = pd.DataFrame(
    [
        ("Separator is ';' (and default read fails)", len(df), "Read with sep=';'"),
        ("Exact duplicate rows", dup_mask.sum(), "Remove duplicates"),
        ("Dates in yyyy-mm-dd format", is_iso.sum(), "Parse both formats explicitly"),
        ("Region spelling variants", (region_fixed != df["Region"]).sum(), "Trim spaces, map to official name"),
        ("Amounts with decimal comma", has_comma.sum(), "Replace ',' with '.' then convert"),
        ("Empty quantities", empty_qty.sum(), "Recompute from amount, price, discount; flag"),
        ("Unknown product or customer IDs",
         (~df["Product ID"].isin(products["ProductID"])).sum()
         + (~df["Customer ID"].isin(customers["CustomerID"])).sum(),
         "None needed; keep the check"),
    ],
    columns=["Issue", "Rows affected", "Planned fix"],
)
summary["% of rows"] = (summary["Rows affected"] / len(df) * 100).round(1)
summary

,Issue,Rows affected,Planned fix,% of rows
0,Separator is ';' (and default read fails),6078,Read with sep=';',100.0
1,Exact duplicate rows,80,Remove duplicates,1.3
2,Dates in yyyy-mm-dd format,927,Parse both formats explicitly,15.3
3,Region spelling variants,501,"Trim spaces, map to official name",8.2
4,Amounts with decimal comma,623,"Replace ',' with '.' then convert",10.3
5,Empty quantities,119,"Recompute from amount, price, discount; flag",2.0
6,Unknown product or customer IDs,0,None needed; keep the check,0.0


**Conclusions**

- The raw exports cannot be analysed as they are: totals by region would be split, duplicates would inflate turnover, and some rows would fail to convert.
- Every problem follows a clear pattern, so each one can be fixed with a simple rule and the whole cleaning can be fully automated. 
- No row has to be deleted except true duplicates: even rows with a missing quantity can be recovered, because the quantity can be recalculated.

**Next step:** implement these rules in `src/sales_report/cleaning.py`, with a test for each one.